# Unidad 19. Selección, filtrado y calidad de los datos

**Duración orientativa:** 7 horas · **Versión:** 1.0 · **UT3**

Un análisis útil empieza por decidir qué datos se necesitan y por comprobar que representan lo que dicen representar. En esta unidad seleccionarás filas y columnas con precisión, detectarás problemas de calidad y aplicarás transformaciones documentadas.

## Resultados de aprendizaje

Al finalizar podrás seleccionar columnas y filas, construir filtros booleanos, localizar valores ausentes y duplicados, normalizar texto, convertir datos numéricos con errores controlados y explicar por qué aplicas cada transformación.

**Conocimientos previos:** Series, DataFrame, rutas relativas, lectura de CSV e inspección inicial de la unidad 18.

**Material:** datos/pandas/19_lecturas_estaciones_calidad.csv. Es un conjunto didáctico con problemas introducidos deliberadamente.

## 1. Seleccionar no es analizar todo

Seleccionar una columna o filtrar filas permite responder una pregunta concreta sin perder la tabla original. Una selección correcta depende de saber qué representa cada variable. Antes de escribir código, formula la pregunta: por ejemplo, qué lecturas pertenecen a una zona o qué identificadores están repetidos.

Pandas distingue la selección por etiqueta de la selección por posición. Las etiquetas son nombres de columnas o etiquetas del índice; las posiciones son números enteros. Mezclarlas sin atención produce errores silenciosos, especialmente cuando el índice ya no coincide con 0, 1, 2 y así sucesivamente.

In [1]:
from pathlib import Path
import pandas as pd

ruta_datos = Path('..') / 'datos' / 'pandas' / '19_lecturas_estaciones_calidad.csv'
if not ruta_datos.exists():
    ruta_datos = Path('datos') / 'pandas' / '19_lecturas_estaciones_calidad.csv'

lecturas_original = pd.read_csv(ruta_datos)
lecturas_original.head()

,id_lectura,fecha,hora,estacion_id,zona,temperatura_c,humedad_pct,pm25_ug_m3,estado_sensor,observaciones
0,L-001,2025-01-13,08:00,EST-01,Norte,10.8,74.0,18.4,activo,lectura ordinaria
1,L-002,2025-01-13,14:00,EST-01,Norte,16.2,52.0,12.7,activo,lectura ordinaria
2,L-003,2025-01-13,20:00,EST-01,norte,12.1,68.0,15.9,Activo,lectura ordinaria
3,L-004,2025-01-13,08:00,EST-02,Centro,11.7,70.0,23.5,activo,calibracion reciente
4,L-005,2025-01-13,14:00,EST-02,Centro,17.4,NaN,19.8,activo,sin humedad


## 2. Columnas, Series y loc

Los corchetes con un nombre devuelven una Series; con una lista de nombres devuelven un DataFrame. Esta diferencia importa porque una Series representa una variable y un DataFrame mantiene una tabla. loc selecciona por etiqueta y admite simultáneamente filas y columnas. iloc selecciona por posición entera.

En ejemplos pequeños puede parecer más rápido usar números, pero los nombres expresivos suelen resistir mejor cambios en el orden de las columnas.

In [2]:
temperaturas = lecturas_original['temperatura_c']
tabla_medidas = lecturas_original[['id_lectura', 'zona', 'temperatura_c', 'pm25_ug_m3']]

print(type(temperaturas).__name__)
print(type(tabla_medidas).__name__)
display(tabla_medidas.head())
display(lecturas_original.loc[0:2, ['id_lectura', 'zona']])
display(lecturas_original.iloc[0:3, 0:3])

Series
DataFrame


,id_lectura,zona,temperatura_c,pm25_ug_m3
0,L-001,Norte,10.8,18.4
1,L-002,Norte,16.2,12.7
2,L-003,norte,12.1,15.9
3,L-004,Centro,11.7,23.5
4,L-005,Centro,17.4,19.8


,id_lectura,zona
0,L-001,Norte
1,L-002,Norte
2,L-003,norte


,id_lectura,fecha,hora
0,L-001,2025-01-13,08:00
1,L-002,2025-01-13,14:00
2,L-003,2025-01-13,20:00


## 3. Filtros booleanos

Una condición como lecturas['zona'] == 'Norte' produce una Series de True y False, llamada máscara booleana. Al usarla entre corchetes se conservan las filas True. Para combinar condiciones usa & para Y, | para O y ~ para NO. Cada comparación necesita sus propios paréntesis: Python no permite encadenar comparaciones de Series como si fueran valores individuales.

Un filtro no corrige datos. Si Norte y norte representan la misma zona pero aparecen escritos de dos maneras, un filtro puede omitir registros. Primero observamos el problema; después decidimos una normalización reproducible.

In [3]:
mascara_temperatura = lecturas_original['temperatura_c'] >= 15
display(lecturas_original[mascara_temperatura][['id_lectura', 'zona', 'temperatura_c']])

filtro_compuesto = (lecturas_original['estacion_id'] == 'EST-02') & (lecturas_original['temperatura_c'] >= 15)
display(lecturas_original.loc[filtro_compuesto, ['id_lectura', 'estacion_id', 'temperatura_c']])

,id_lectura,zona,temperatura_c
1,L-002,Norte,16.2
4,L-005,Centro,17.4
8,L-008,Sur,18.3
11,L-011,Norte,15.8
14,L-014,Centro,16.9


,id_lectura,estacion_id,temperatura_c
4,L-005,EST-02,17.4
14,L-014,EST-02,16.9


## 4. Calidad de datos: observar antes de decidir

La calidad no significa que todos los valores sean perfectos. Significa conocer qué problemas existen, qué impacto tienen y qué decisión se toma. Un valor ausente puede indicar que no se midió, que no se conocía o que el sistema falló. Una fila duplicada puede ser una repetición accidental o un evento real con la misma información.

No hay una receta universal como borrar todas las filas con ausencias. La decisión depende de la finalidad del análisis, de la proporción afectada y del significado de la variable. En este conjunto didáctico conocemos los problemas porque se han introducido para practicar; en datos reales habrá que investigarlos.

In [4]:
print('Valores ausentes por columna:')
print(lecturas_original.isna().sum())
print('\nFilas duplicadas completas:', lecturas_original.duplicated().sum())
print('Identificadores repetidos:')
print(lecturas_original['id_lectura'].value_counts().loc[lambda serie: serie > 1])
print('\nZonas observadas:')
print(lecturas_original['zona'].value_counts(dropna=False))
print('\nTipos inferidos:')
print(lecturas_original.dtypes)

Valores ausentes por columna:
id_lectura       0
fecha            0
hora             0
estacion_id      0
zona             0
temperatura_c    0
humedad_pct      1
pm25_ug_m3       0
estado_sensor    0
observaciones    0
dtype: int64

Filas duplicadas completas: 1
Identificadores repetidos:
id_lectura
L-006    2
Name: count, dtype: int64

Zonas observadas:
zona
Centro      6
Norte       5
Sur         3
norte       1
 Centro     1
Name: count, dtype: int64

Tipos inferidos:
id_lectura           str
fecha                str
hora                 str
estacion_id          str
zona                 str
temperatura_c    float64
humedad_pct      float64
pm25_ug_m3           str
estado_sensor        str
observaciones        str
dtype: object


### Ausencias y duplicados

isna localiza ausencias y sum cuenta cuántas hay. duplicated sin argumentos detecta filas completas repetidas; duplicated con subset permite decidir qué columnas identifican una observación. Antes de eliminar duplicados, guarda la regla elegida. En este caso, id_lectura debe ser único y la fila L-006 se repite íntegramente, por lo que eliminar la segunda copia está justificado.

dropna elimina filas o columnas con ausencias; fillna las sustituye. Ambas operaciones cambian los datos y requieren una razón. Para la humedad ausente usaremos la mediana solo como práctica explícita: en un análisis real deberías valorar si esa imputación es apropiada.

## 5. Crear una versión limpia sin perder el original

Evita sobrescribir el DataFrame original durante una exploración. copy crea una tabla independiente donde puedes registrar transformaciones. Cada paso siguiente responde a un problema observado: espacios y mayúsculas inconsistentes en zona, coma decimal en pm25_ug_m3, ausencia de humedad y fila duplicada.

La limpieza no debe ocultar lo ocurrido. En un proyecto completo se conservaría un registro de decisiones y, si fuera necesario, las filas eliminadas para auditoría.

In [5]:
lecturas_limpias = lecturas_original.copy()

# Normalizamos texto: eliminamos espacios exteriores y unificamos mayúscula inicial.
lecturas_limpias['zona'] = lecturas_limpias['zona'].str.strip().str.title()
lecturas_limpias['estado_sensor'] = lecturas_limpias['estado_sensor'].str.strip().str.lower()

# Convertimos una coma decimal en punto y los errores en ausencia explícita.
lecturas_limpias['pm25_ug_m3'] = pd.to_numeric(
    lecturas_limpias['pm25_ug_m3'].str.replace(',', '.', regex=False),
    errors='coerce',
)

# La mediana se calcula antes de rellenar la ausencia.
mediana_humedad = lecturas_limpias['humedad_pct'].median()
lecturas_limpias['humedad_pct'] = lecturas_limpias['humedad_pct'].fillna(mediana_humedad)

# Conservamos la primera aparición de cada identificador.
lecturas_limpias = lecturas_limpias.drop_duplicates(subset='id_lectura', keep='first').copy()
lecturas_limpias.info()

<class 'pandas.DataFrame'>
Index: 15 entries, 0 to 15
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_lectura     15 non-null     str    
 1   fecha          15 non-null     str    
 2   hora           15 non-null     str    
 3   estacion_id    15 non-null     str    
 4   zona           15 non-null     str    
 5   temperatura_c  15 non-null     float64
 6   humedad_pct    15 non-null     float64
 7   pm25_ug_m3     15 non-null     float64
 8   estado_sensor  15 non-null     str    
 9   observaciones  15 non-null     str    
dtypes: float64(3), str(7)
memory usage: 1.3 KB


El uso de errors='coerce' no arregla mágicamente un valor: convierte los valores que no puede interpretar en ausencias. Es preferible a obtener un error oculto o a convertir una columna incorrectamente. Tras una conversión hay que comprobar cuántas ausencias han aparecido y decidir cómo tratarlas.

Observa también que assign, copy y las operaciones de texto devuelven nuevos objetos o columnas. Evitar modificaciones ambiguas reduce avisos y hace más fácil razonar sobre el código.

In [6]:
print('Forma original:', lecturas_original.shape)
print('Forma limpia:', lecturas_limpias.shape)
print('Ausencias tras limpieza:')
print(lecturas_limpias.isna().sum())
print('Zonas normalizadas:', sorted(lecturas_limpias['zona'].unique()))
print('Tipo de pm25_ug_m3:', lecturas_limpias['pm25_ug_m3'].dtype)
assert lecturas_limpias['id_lectura'].is_unique
assert lecturas_limpias['humedad_pct'].notna().all()

Forma original: (16, 10)
Forma limpia: (15, 10)
Ausencias tras limpieza:
id_lectura       0
fecha            0
hora             0
estacion_id      0
zona             0
temperatura_c    0
humedad_pct      0
pm25_ug_m3       0
estado_sensor    0
observaciones    0
dtype: int64
Zonas normalizadas: ['Centro', 'Norte', 'Sur']
Tipo de pm25_ug_m3: float64


## 6. Transformar columnas de forma explícita

Una nueva columna debe tener un nombre que exprese su unidad o significado. Aquí convertimos temperatura de grados Celsius a Fahrenheit usando una operación vectorizada: Pandas aplica la fórmula a toda la Series. Las operaciones vectorizadas suelen ser más claras y eficientes que recorrer fila a fila.

Las reglas de clasificación también deben declarar qué ocurre en los límites. Usaremos una condición sencilla para etiquetar una concentración como elevada cuando sea igual o superior a 20 microgramos por metro cúbico. No es una norma sanitaria: es una regla didáctica para practicar transformaciones.

In [7]:
lecturas_limpias['temperatura_f'] = lecturas_limpias['temperatura_c'] * 9 / 5 + 32
lecturas_limpias['pm25_elevada'] = lecturas_limpias['pm25_ug_m3'] >= 20
lecturas_limpias[['id_lectura', 'temperatura_c', 'temperatura_f', 'pm25_ug_m3', 'pm25_elevada']].head()

,id_lectura,temperatura_c,temperatura_f,pm25_ug_m3,pm25_elevada
0,L-001,10.8,51.44,18.4,False
1,L-002,16.2,61.16,12.7,False
2,L-003,12.1,53.78,15.9,False
3,L-004,11.7,53.06,23.5,True
4,L-005,17.4,63.32,19.8,False


## Errores frecuentes

- Usar and u or con Series: utiliza & y |, y pon cada condición entre paréntesis.
- Filtrar antes de normalizar categorías: puede excluir variantes como Norte, norte o espacios exteriores.
- Borrar ausencias sin explicar su significado: podrías sesgar el resultado o perder información relevante.
- Convertir tipos con astype sin comprobar errores: una cadena como 21,2 no se interpreta automáticamente como número decimal.
- Modificar la tabla original y no poder comparar antes y después: conserva una copia o asigna a una nueva variable.
- Confundir una regla didáctica con una recomendación real: una etiqueta creada para practicar no equivale a una evaluación oficial.

## Práctica guiada

1. Localiza la fila repetida antes de ejecutar la limpieza.
2. Comprueba por qué zona tiene más valores distintos de los esperados antes de normalizar.
3. Explica por qué la columna pm25_ug_m3 se carga como texto.
4. Ejecuta la limpieza por pasos y compara shape antes y después.
5. Cambia temporalmente el umbral de pm25_elevada a 25 y observa qué filas cambian.

## Ejercicios

Realiza los ejercicios usando una copia de lecturas_original. Las soluciones se proporcionan en un cuaderno independiente.

**Ejercicio 19.01.** Crea un DataFrame con id_lectura, zona y temperatura_c. Selecciona mediante loc las tres primeras filas de esas columnas y explica la diferencia con iloc.

In [8]:
# Tu solución del ejercicio 19.01
df = pd.DataFrame({
    "id_lectura": ["A-01","A-02","A-03","A-04",],
    "zona": ["Madrid", "Valencia", "Santander", "Murcia"],
    "temperatura_c": [25.5, 28.2, 19.7, 14]
})

print(df.loc[0:2] )
# iloc busca por posiciones numericas.

  id_lectura       zona  temperatura_c
0       A-01     Madrid           25.5
1       A-02   Valencia           28.2
2       A-03  Santander           19.7


**Ejercicio 19.02.** Filtra las lecturas de EST-02 cuya temperatura sea mayor o igual que 15. Antes de filtrar, escribe qué columnas necesitas y por qué.

In [9]:
# Tu solución del ejercicio 19.02
ruta_datos = Path("../datos/pandas/19_lecturas_estaciones_calidad.csv")
df = pd.read_csv(ruta_datos)
print(df.loc[(df["estacion_id"] == "EST-02") & (df["temperatura_c"] >= 15)])


   id_lectura       fecha   hora estacion_id    zona  temperatura_c  \
4       L-005  2025-01-13  14:00      EST-02  Centro           17.4   
14      L-014  2025-01-14  14:00      EST-02  Centro           16.9   

    humedad_pct pm25_ug_m3 estado_sensor      observaciones  
4           NaN       19.8        activo        sin humedad  
14         54.0       22.4        activo  lectura ordinaria  


**Ejercicio 19.03.** Elabora una tabla de diagnóstico con ausencias por columna, número de filas duplicadas y frecuencias de zona. Identifica los cuatro problemas intencionados del CSV.

In [10]:
# Tu solución del ejercicio 19.03
print(df.isna().sum())
print("Suma de duplicados")
print(df.duplicated().sum())
print("Frecuencias de zona")
print(df["zona"].value_counts(dropna = False))

id_lectura       0
fecha            0
hora             0
estacion_id      0
zona             0
temperatura_c    0
humedad_pct      1
pm25_ug_m3       0
estado_sensor    0
observaciones    0
dtype: int64
Suma de duplicados
1
Frecuencias de zona
zona
Centro      6
Norte       5
Sur         3
norte       1
 Centro     1
Name: count, dtype: int64


Errores intencionados:
Espacios sobrantes y diferencias de mayúsculas/minúsculas en zona y estado_sensor.
Un dato faltante en la columna humedad_pct
La lectura L-006 aparece repetida dos veces
Uso de coma decimal ("21,2") en pm25_ug_m3, convirtiendo la columna a texto.

**Ejercicio 19.04.** Crea una versión limpia que normalice zona, convierta pm25_ug_m3 a número, complete humedad_pct con la mediana y elimine la repetición por id_lectura. Añade comprobaciones que justifiquen el resultado.

In [11]:
# Tu solución del ejercicio 19.04
df_limpio = df.copy()
df_limpio['zona'] = df_limpio['zona'].str.strip().str.title()
df_limpio['pm25_ug_m3'] = pd.to_numeric(
    df_limpio['pm25_ug_m3'].str.replace(',', '.', regex = False),
    errors= "coerce"
)
mediana_humedad = df['humedad_pct'].median()
df_limpio['humedad_pct'] = df_limpio['humedad_pct'].fillna(mediana_humedad)
df_limpio = df_limpio.drop_duplicates(subset='id_lectura', keep='first')
df_limpio

,id_lectura,fecha,hora,estacion_id,zona,temperatura_c,humedad_pct,pm25_ug_m3,estado_sensor,observaciones
0,L-001,2025-01-13,08:00,EST-01,Norte,10.8,74.0,18.4,activo,lectura ordinaria
1,L-002,2025-01-13,14:00,EST-01,Norte,16.2,52.0,12.7,activo,lectura ordinaria
2,L-003,2025-01-13,20:00,EST-01,Norte,12.1,68.0,15.9,Activo,lectura ordinaria
3,L-004,2025-01-13,08:00,EST-02,Centro,11.7,70.0,23.5,activo,calibracion reciente
4,L-005,2025-01-13,14:00,EST-02,Centro,17.4,66.0,19.8,activo,sin humedad
5,L-006,2025-01-13,20:00,EST-02,Centro,13.3,65.0,21.2,activo,separador decimal incorrecto
7,L-007,2025-01-13,08:00,EST-03,Sur,12.5,66.0,14.1,activo,lectura ordinaria
8,L-008,2025-01-13,14:00,EST-03,Sur,18.3,43.0,10.6,activo,lectura ordinaria
9,L-009,2025-01-13,20:00,EST-03,Sur,14.0,61.0,12.9,activo,lectura ordinaria
10,L-010,2025-01-14,08:00,EST-01,Norte,9.9,79.0,20.3,activo,lectura ordinaria


**Ejercicio 19.05.** Añade duracion_turno, una columna textual que valga mañana para horas anteriores a 12 y tarde en los demás casos. Justifica qué tipo tiene hora y por qué en esta unidad puedes compararla como texto con formato HH:MM.

In [20]:
# Tu solución del ejercicio 19.05
import numpy as np
print(df['hora'].head())
print(df_limpio['hora'].dtype)
df_limpio['duracion_turno'] = np.where(df_limpio['hora']< '12:00', 'Mañana', 'Tarde')

# Pandas devuelve un objeto que almacena texto
# Las horas tienen un formato HH:MM
# Dado que estan en tipo str se puede comparar la cadena de texto

0    08:00
1    14:00
2    20:00
3    08:00
4    14:00
Name: hora, dtype: str
str


## Autoevaluación

1. ¿Cuándo devuelve una selección una Series y cuándo un DataFrame?
2. ¿Por qué & requiere paréntesis en filtros compuestos?
3. ¿Qué diferencia hay entre observar una ausencia, imputarla y eliminarla?
4. ¿Por qué conviene conservar lecturas_original?
5. ¿Qué comprobación demuestra que cada id_lectura aparece una sola vez?

La unidad 20 utilizará datos ya preparados para resumir, agrupar y ordenar. Antes de avanzar, asegúrate de poder explicar cada regla de limpieza aplicada aquí.

## Recursos de consulta

- Selección e indexación: https://pandas.pydata.org/docs/user_guide/indexing.html
- Datos ausentes: https://pandas.pydata.org/docs/user_guide/missing_data.html
- Operaciones de texto: https://pandas.pydata.org/docs/user_guide/text.html